# Big Data Analysis and Application Experiment

**Experiment Topic:** Data Preparation with pandas  
**School:** Hubei University  
**Class:** Software Engineering 2402  
**Student Name:** Qin Tian  
**Student ID:** 21907869  
**Instructor:** Li Jie

# Part 1. Problem Analysis

This experiment is about employee attrition data. The target is not building the final model right now. Emm, this step is more like a prep step. I need to clean the raw data and make the train set and test set ready for the next logistic regression task.

The train file has the label column `Attrition`, but the test file does not. So the main work is pretty direct: check the table, find useless columns, encode text columns, scale numeric columns, and keep the final feature format the same in both datasets.

# Part 2. Data Description

There are two source files in this experiment: `pfm_train.csv` and `pfm_test.csv`. The train set is used to learn the later model, and the test set is used for prediction.

Before doing any cleaning, I need to know some very basic things, such as the number of rows and columns, the column types, whether there are missing values, and whether there are duplicated rows. Oh, this part looks simple, but it actually decides the later processing way.

# Part 3. Processing Method

The method in this notebook is not complicated. First I inspect the raw data. Then I remove columns that do not help, such as constant columns and the id column. After that, I use one-hot encoding for categorical features.

For numeric features, I apply standardization by `StandardScaler`. The scaler is fitted on the train set first, then the same rule is used on the test set. This is a cleaner way, because the test set should not affect the scaling rule.

# Part 4. Source Code and Process

## 4.1 Read the raw csv files

First thing first, I load the two csv files and take a quick look at the train set.

In [1]:
# use pandas to read the csv files
import pandas as pd

# this one is for scaling the numeric features later
from sklearn.preprocessing import StandardScaler

# read train data and test data
train = pd.read_csv("pfm_train.csv")
test = pd.read_csv("pfm_test.csv")

# just check the size first
print("train shape:", train.shape)
print("test shape:", test.shape)

# have a look of the first few rows
train.head()

train shape: (1100, 31)
test shape: (350, 30)


,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,Gender,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,37,0,Travel_Rarely,Research & Development,1,4,Life Sciences,77,1,Male,...,3,80,1,7,2,4,7,5,0,7
1,54,0,Travel_Frequently,Research & Development,1,4,Life Sciences,1245,4,Female,...,1,80,1,33,2,1,5,4,1,4
2,34,1,Travel_Frequently,Research & Development,7,3,Life Sciences,147,1,Male,...,4,80,0,9,3,3,9,7,0,6
3,39,0,Travel_Rarely,Research & Development,1,1,Life Sciences,1026,4,Female,...,3,80,1,21,3,3,21,6,11,8
4,28,1,Travel_Frequently,Research & Development,1,3,Medical,1111,1,Male,...,1,80,2,1,2,3,1,0,0,0


## 4.2 Check basic information and missing values

Here I check the column types and missing values. If there are too many null values, the cleaning plan may change a lot.

In [2]:
# check basic info of train set
print("=== train info ===")
train.info()

# also check the test set
print("\n=== test info ===")
test.info()

# see if there is any missing values in train
print("\n=== missing values in train ===")
print(train.isnull().sum())

# same thing for test
print("\n=== missing values in test ===")
print(test.isnull().sum())

=== train info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1100 non-null   int64 
 1   Attrition                 1100 non-null   int64 
 2   BusinessTravel            1100 non-null   object
 3   Department                1100 non-null   object
 4   DistanceFromHome          1100 non-null   int64 
 5   Education                 1100 non-null   int64 
 6   EducationField            1100 non-null   object
 7   EmployeeNumber            1100 non-null   int64 
 8   EnvironmentSatisfaction   1100 non-null   int64 
 9   Gender                    1100 non-null   object
 10  JobInvolvement            1100 non-null   int64 
 11  JobLevel                  1100 non-null   int64 
 12  JobRole                   1100 non-null   object
 13  JobSatisfaction           1100 non-null   int64 
 14  Marit

## 4.3 Check duplicate rows and constant columns

If a row is repeated many times, it may affect the later model. Also, if one column has only one value, it does not give real information.

In [3]:
# check duplicate rows first
print("duplicate rows in train:", train.duplicated().sum())
print("duplicate rows in test:", test.duplicated().sum())

# find the columns which only have one value
const_train = [col for col in train.columns if train[col].nunique(dropna=False) == 1]
const_test = [col for col in test.columns if test[col].nunique(dropna=False) == 1]

# print them out
print("constant cols in train:", const_train)
print("constant cols in test:", const_test)

duplicate rows in train: 0
duplicate rows in test: 0
constant cols in train: ['Over18', 'StandardHours']
constant cols in test: ['Over18', 'StandardHours']


## 4.4 Check the label and categorical columns

I also want to see the label distribution and the raw text columns. This part helps me know what kind of encoding is needed.

In [4]:
# see the target label distribution
print("attrition counts:")
print(train["Attrition"].value_counts())

# find all categorical columns in raw data
cat_cols_raw = train.select_dtypes(include="object").columns.tolist()
print("\nraw categorical cols:", cat_cols_raw)

# check each category values
for col in cat_cols_raw:
    print(f"\nvalue counts of {col}:")
    print(train[col].value_counts())

attrition counts:
Attrition
0    922
1    178
Name: count, dtype: int64

raw categorical cols: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'Over18', 'OverTime']

value counts of BusinessTravel:
BusinessTravel
Travel_Rarely        787
Travel_Frequently    205
Non-Travel           108
Name: count, dtype: int64

value counts of Department:
Department
Research & Development    727
Sales                     331
Human Resources            42
Name: count, dtype: int64

value counts of EducationField:
EducationField
Life Sciences       462
Medical             337
Marketing           127
Technical Degree     92
Other                63
Human Resources      19
Name: count, dtype: int64

value counts of Gender:
Gender
Male      653
Female    447
Name: count, dtype: int64

value counts of JobRole:
JobRole
Sales Executive              247
Research Scientist           221
Laboratory Technician        205
Manufacturing Director       101
Healthcare Represen

## 4.5 Drop useless columns

After the checks above, I remove `Over18`, `StandardHours`, and `EmployeeNumber`. The first two are constant, and the last one is only an employee id.

In [5]:
# copy the data first, so the raw one not be changed
train_clean = train.copy()
test_clean = test.copy()

# these columns are not useful, so remove them
# Over18 and StandardHours are basically same value
# EmployeeNumber is just id
drop_cols = ["Over18", "StandardHours", "EmployeeNumber"]

# drop these columns in both train and test
train_clean = train_clean.drop(columns=drop_cols)
test_clean = test_clean.drop(columns=drop_cols)

# check the shape after dropping
print("train_clean shape:", train_clean.shape)
print("test_clean shape:", test_clean.shape)

train_clean shape: (1100, 28)
test_clean shape: (350, 27)


## 4.6 Encode categorical features

Text columns can not go into a simple model directly, so I turn them into dummy columns. I keep `drop_first=True` here, because it makes the final table a bit shorter.

In [6]:
# get the categorical columns after dropping useless ones
cat_cols = train_clean.select_dtypes(include="object").columns.tolist()
print("categorical cols after drop:", cat_cols)

# do one-hot encoding for categorical features
# drop_first=True to avoid some repeated dummy columns
train_encoded = pd.get_dummies(train_clean, columns=cat_cols, drop_first=True)
test_encoded = pd.get_dummies(test_clean, columns=cat_cols, drop_first=True)

# split x and y from training data
X_train = train_encoded.drop(columns="Attrition")
y_train = train_encoded["Attrition"]

# make test columns same with train columns
# if some column is missing in test, fill it by 0
X_test = test_encoded.reindex(columns=X_train.columns, fill_value=0)

# find bool type columns
bool_cols_train = X_train.select_dtypes(include="bool").columns
bool_cols_test = X_test.select_dtypes(include="bool").columns

# change bool to int, maybe better for model later
X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)
X_test[bool_cols_test] = X_test[bool_cols_test].astype(int)

# check shape after encoding
print("encoded train shape:", train_encoded.shape)
print("encoded test shape:", test_encoded.shape)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

categorical cols after drop: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
encoded train shape: (1100, 42)
encoded test shape: (350, 41)
X_train shape: (1100, 41)
X_test shape: (350, 41)


## 4.7 Scale numeric features

Emm, here I only scale the original numeric columns. The dummy columns are already 0 and 1, so I leave them like that.

In [7]:
# pick numeric columns for scaling
# do not include the target column
num_cols = train_clean.select_dtypes(exclude="object").columns.tolist()
num_cols.remove("Attrition")

# print these numeric columns
print("scaled numeric cols:", num_cols)

# create the scaler
scaler = StandardScaler()

# fit on train data first, then transform it
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# use the same scaler on test data
X_test[num_cols] = scaler.transform(X_test[num_cols])

# look the processed train data
X_train.head()

scaled numeric cols: ['Age', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


,Age,DistanceFromHome,Education,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,NumCompaniesWorked,PercentSalaryHike,...,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes
0,0.000101,-1.028598,1.054313,-1.572091,-1.035216,-0.049260,0.240954,-0.104096,-0.671072,0.762229,...,0,0,1,0,0,0,0,0,0,0
1,1.882064,-1.028598,1.054313,1.161260,0.381124,0.853837,0.240954,0.852589,1.720437,0.486513,...,0,0,1,0,0,0,0,0,0,0
2,-0.332010,-0.296263,0.075626,-1.572091,-2.451557,-0.049260,0.240954,-0.086910,-0.671072,2.416525,...,1,0,0,0,0,0,0,0,1,1
3,0.221508,-1.028598,-1.881748,1.161260,-1.035216,1.756934,1.142483,1.327855,-0.671072,0.210797,...,0,0,1,0,0,0,0,1,0,0
4,-0.996233,-1.028598,0.075626,-1.572091,-1.035216,-0.952357,-0.660575,-0.824846,-0.671072,-0.064919,...,1,0,0,0,0,0,0,0,0,0


## 4.8 Final check of prepared data

At last, I check the final table shape and make sure there is no missing value left.

In [8]:
# pick numeric columns for scaling
# do not include the target column
num_cols = train_clean.select_dtypes(exclude="object").columns.tolist()
num_cols.remove("Attrition")

# print these numeric columns
print("scaled numeric cols:", num_cols)

# create the scaler
scaler = StandardScaler()

# fit on train data first, then transform it
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# use the same scaler on test data
X_test[num_cols] = scaler.transform(X_test[num_cols])

# look the processed train data
X_train.head()

scaled numeric cols: ['Age', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


,Age,DistanceFromHome,Education,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,NumCompaniesWorked,PercentSalaryHike,...,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes
0,0.000101,-1.028598,1.054313,-1.572091,-1.035216,-0.049260,0.240954,-0.104096,-0.671072,0.762229,...,0,0,1,0,0,0,0,0,0,0
1,1.882064,-1.028598,1.054313,1.161260,0.381124,0.853837,0.240954,0.852589,1.720437,0.486513,...,0,0,1,0,0,0,0,0,0,0
2,-0.332010,-0.296263,0.075626,-1.572091,-2.451557,-0.049260,0.240954,-0.086910,-0.671072,2.416525,...,1,0,0,0,0,0,0,0,1,1
3,0.221508,-1.028598,-1.881748,1.161260,-1.035216,1.756934,1.142483,1.327855,-0.671072,0.210797,...,0,0,1,0,0,0,0,1,0,0
4,-0.996233,-1.028598,0.075626,-1.572091,-1.035216,-0.952357,-0.660575,-0.824846,-0.671072,-0.064919,...,1,0,0,0,0,0,0,0,0,0


# Part 5. Result Analysis and Reflection

After the whole preparation work, the raw data becomes ready for the next model step. The final `X_train` has 1100 rows and 41 columns, `y_train` has 1100 labels, and `X_test` has 350 rows and 41 columns. There is no missing value left in the final feature tables.

From this experiment, I feel data preparation is not a small thing at all. At first it looks easy, but when I really do it step by step, I can see that every check matters, such as missing values, duplicate rows, constant columns, and text encoding. If this part is done carelessly, the next model result may look strange even when the code still runs.

I also learned that train data and test data should be processed in the same format. Oh, this point is very important. For example, the scaler must learn from the train set first, and then use the same rule on the test set. So yeah, this experiment helped me understand that clean data is the real starting point of later data mining work.

In [9]:
# final check of the shapes
print("final X_train shape:", X_train.shape)
print("final y_train shape:", y_train.shape)
print("final X_test shape:", X_test.shape)

# see if there is still missing values
print("missing values in X_train:", X_train.isnull().sum().sum())
print("missing values in X_test:", X_test.isnull().sum().sum())

# check the final data types
print("\nX_train dtype summary:")
print(X_train.dtypes.value_counts())

final X_train shape: (1100, 41)
final y_train shape: (1100,)
final X_test shape: (350, 41)
missing values in X_train: 0
missing values in X_test: 0

X_train dtype summary:
int64      21
float64    20
Name: count, dtype: int64
